# Single-Node Inference

Establish a controlled denominator for all later distributed comparisons.

## Objectives

- Launch or connect to one server and verify model identity and readiness.
- Measure startup separately from steady-state latency and throughput.
- Sweep controlled prompt and output lengths while recording memory and correctness.
- Retain raw per-request results, including failures, before aggregation.

## Background

A single-node baseline defines the denominator needed to interpret distributed results. Startup, warm-up, and steady state require separate measurement boundaries.

## Prediction

For one request at a time on a single DGX Spark, inference behavior should differ between prompt processing and autoregressive generation.

Specifically:

- increasing prompt length while keeping generated length fixed should primarily increase time to first token;
- increasing generated length while keeping prompt length fixed should primarily increase total latency;
- output-token throughput should be relatively stable across sufficiently long generations, but short generations should show lower apparent throughput because fixed request and scheduling overheads make up a larger fraction of total latency;
- the first successful request after server readiness should be slower than later steady-state requests because runtime initialization, kernel loading, graph construction, cache population, or similar one-time work may still occur;
- repeated deterministic requests with the same prompt and sampling configuration should return the same token sequence unless the serving stack introduces nondeterminism;
- no request in the initial single-request benchmark should fail or exceed the configured timeout.

These predictions are falsified if prompt length has no measurable relationship with time to first token, generated length has no measurable relationship with total latency, warm-up requests are not distinguishable from measured requests, or nominally deterministic repeated requests return inconsistent token sequences.

## Environment

In [42]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference


## Experiment

Complete configuration placeholders before running active measurement cells.

### Benchmark configuration and deterministic prompts

In [43]:
from dataclasses import asdict, dataclass
from itertools import product

import pandas as pd


MODEL_ID = "google/gemma-4-E4B-it"
MODEL_REVISION = "ee0ef6023621cff504d758262d4e04895a5af4a2"

HOST = "127.0.0.1"
PORT = 8000
ENDPOINT = f"http://{HOST}:{PORT}"
MODEL = MODEL_ID

# Start with an externally managed server. This keeps server launch and
# inference measurement separate during the first experiment.
SERVER_MODE = "notebook_subprocess"

# Initial factorial workload:
#
# - 32 tokens represents a small interactive prompt.
# - 512 tokens exposes a meaningful prefill difference without making the
#   first experiment unnecessarily expensive.
# - 16 generated tokens emphasizes fixed request overhead.
# - 128 generated tokens provides a more useful decode-throughput interval.
PROMPT_TOKEN_COUNTS = (32, 512)
GENERATED_TOKEN_COUNTS = (16, 128)

WARMUP_COUNT = 3
REPETITIONS = 5
REQUEST_TIMEOUT_S = 300.0

SAMPLING = {
    "temperature": 0.0,
    "top_p": 1.0,
    "seed": 20260806,
}


@dataclass(frozen=True, slots=True)
class BenchmarkConfiguration:
    endpoint: str
    model: str
    model_revision: str
    server_mode: str
    prompt_token_counts: tuple[int, ...]
    generated_token_counts: tuple[int, ...]
    warmup_count: int
    repetitions: int
    request_timeout_s: float
    temperature: float
    top_p: float
    seed: int


benchmark_configuration = BenchmarkConfiguration(
    endpoint=ENDPOINT,
    model=MODEL,
    model_revision=MODEL_REVISION,
    server_mode=SERVER_MODE,
    prompt_token_counts=PROMPT_TOKEN_COUNTS,
    generated_token_counts=GENERATED_TOKEN_COUNTS,
    warmup_count=WARMUP_COUNT,
    repetitions=REPETITIONS,
    request_timeout_s=REQUEST_TIMEOUT_S,
    temperature=SAMPLING["temperature"],
    top_p=SAMPLING["top_p"],
    seed=SAMPLING["seed"],
)

pd.Series(asdict(benchmark_configuration), name="value")

endpoint                                     http://127.0.0.1:8000
model                                        google/gemma-4-E4B-it
model_revision            ee0ef6023621cff504d758262d4e04895a5af4a2
server_mode                                    notebook_subprocess
prompt_token_counts                                      (32, 512)
generated_token_counts                                   (16, 128)
warmup_count                                                     3
repetitions                                                      5
request_timeout_s                                            300.0
temperature                                                    0.0
top_p                                                          1.0
seed                                                      20260806
Name: value, dtype: object

### Measurement contract

This notebook uses one request at a time. It does not measure batching, concurrency, queueing, or distributed execution.

For each request:

- **Prompt tokens** are the input token count reported by the server, checked against the intended workload size.
- **Output tokens** are the generated token count reported by the server.
- **Time to first token (TTFT)** is the elapsed monotonic wall-clock time from immediately before request submission until the first non-empty streamed token or text fragment is received.
- **End-to-end latency** is the elapsed monotonic wall-clock time from immediately before request submission until the complete response stream has been consumed.
- **Generation interval** is `end-to-end latency - TTFT`.
- **Output-token throughput** is the number of returned output tokens divided by the generation interval.
- **End-to-end token throughput** is the number of returned output tokens divided by end-to-end latency.

Warm-up requests are retained separately and are never included in steady-state aggregates.

All raw request records, including failed requests and token-count mismatches, are retained before aggregation.

In [44]:
def validate_benchmark_configuration(
    configuration: BenchmarkConfiguration,
) -> None:
    if not configuration.endpoint.startswith(("http://", "https://")):
        raise ValueError("endpoint must be an HTTP or HTTPS URL")

    if not configuration.model:
        raise ValueError("model must not be empty")

    if configuration.server_mode not in {"external", "notebook_subprocess"}:
        raise ValueError("server_mode must be 'external' or 'notebook_subprocess'")

    for name, values in (
        ("prompt_token_counts", configuration.prompt_token_counts),
        ("generated_token_counts", configuration.generated_token_counts),
    ):
        if not values:
            raise ValueError(f"{name} must not be empty")
        if any(
            isinstance(value, bool) or not isinstance(value, int) or value <= 0
            for value in values
        ):
            raise ValueError(f"{name} must contain positive integers")
        if len(values) != len(set(values)):
            raise ValueError(f"{name} must not contain duplicates")

    if configuration.warmup_count < 0:
        raise ValueError("warmup_count must be non-negative")

    if configuration.repetitions <= 0:
        raise ValueError("repetitions must be positive")

    if configuration.request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be positive")

    if configuration.temperature < 0:
        raise ValueError("temperature must be non-negative")

    if not 0 < configuration.top_p <= 1:
        raise ValueError("top_p must be in the interval (0, 1]")


validate_benchmark_configuration(benchmark_configuration)

workload_matrix = pd.DataFrame(
    [
        {
            "prompt_tokens_target": prompt_tokens,
            "requested_output_tokens": generated_tokens,
        }
        for prompt_tokens, generated_tokens in product(
            PROMPT_TOKEN_COUNTS,
            GENERATED_TOKEN_COUNTS,
        )
    ]
)

expected_measured_requests = len(workload_matrix) * REPETITIONS

print(f"Warm-up requests: {WARMUP_COUNT}")
print(f"Workload configurations: {len(workload_matrix)}")
print(f"Repetitions per configuration: {REPETITIONS}")
print(f"Expected measured requests: {expected_measured_requests}")

workload_matrix

Warm-up requests: 3
Workload configurations: 4
Repetitions per configuration: 5
Expected measured requests: 20


,prompt_tokens_target,requested_output_tokens
0,32,16
1,32,128
2,512,16
3,512,128


### Exact-length prompt construction

Prompt lengths must be established with the tokenizer belonging to the pinned model revision.

The notebook environment does not assume that `transformers` is installed. Tokenization therefore runs in the previously verified vLLM container with the host Hugging Face cache mounted read-only.

The generated prompts use repeated neutral prose. Their purpose is to produce controlled token counts, not to evaluate factual knowledge or instruction-following quality.

The target count includes any tokens added by the model's chat template because those tokens are part of the server's actual prefill input.

In [45]:
import json
import shlex
import subprocess
from pathlib import Path


def read_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}

    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        key, separator, value = line.partition("=")
        if not separator:
            raise ValueError(f"Invalid configuration line: {raw_line!r}")

        values[key.strip()] = value.strip()

    return values


cluster = read_env_file(repository_root / "config" / "cluster.env")

VLLM_IMAGE = cluster["VLLM_IMAGE"]
HF_CACHE_ROOT = Path.home() / ".cache" / "huggingface"
MODEL_CACHE_DIRECTORY = "models--google--gemma-4-E4B-it"
MODEL_SNAPSHOT_PATH = (
    HF_CACHE_ROOT / "hub" / MODEL_CACHE_DIRECTORY / "snapshots" / MODEL_REVISION
)

if not MODEL_SNAPSHOT_PATH.is_dir():
    raise FileNotFoundError(
        f"Pinned model snapshot is not available: {MODEL_SNAPSHOT_PATH}"
    )

tokenizer_environment = pd.Series(
    {
        "vllm_image": VLLM_IMAGE,
        "host_cache_root": str(HF_CACHE_ROOT),
        "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
        "model_revision": MODEL_REVISION,
    },
    name="value",
)

tokenizer_environment

vllm_image                                                     vllm-node
host_cache_root                           /home/coert/.cache/huggingface
model_snapshot_path    /home/coert/.cache/huggingface/hub/models--goo...
model_revision                  ee0ef6023621cff504d758262d4e04895a5af4a2
Name: value, dtype: str

In [46]:
import json
import subprocess
import tempfile


TOKENIZER_SCRIPT = r"""
import json
import sys

from transformers import AutoTokenizer


configuration_path = sys.argv[1]
with open(configuration_path, encoding="utf-8") as file:
    configuration = json.load(file)

tokenizer = AutoTokenizer.from_pretrained(
    configuration["model_path"],
    local_files_only=True,
    trust_remote_code=False,
)

seed_text = (
    "The system processes a fixed sequence of neutral words for a controlled "
    "inference benchmark. Each sentence contains ordinary language and adds "
    "no request for external facts. "
)

instruction = (
    "Continue the following text with a concise neutral sentence. "
    "Do not use headings or lists."
)


def serialize_messages(content: str) -> str:
    messages = [
        {
            "role": "user",
            "content": f"{instruction}\n\n{content}",
        }
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def encode(text: str) -> list[int]:
    return tokenizer.encode(text, add_special_tokens=False)


def make_exact_prompt(target_tokens: int) -> dict:
    content = ""

    while len(encode(serialize_messages(content))) < target_tokens:
        content += seed_text

    serialized = serialize_messages(content)
    serialized_ids = encode(serialized)
    exact_ids = serialized_ids[:target_tokens]

    prompt = tokenizer.decode(
        exact_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    verified_ids = encode(prompt)

    if verified_ids != exact_ids:
        raise RuntimeError(
            f"Target {target_tokens}: decoded prompt does not reproduce "
            "the selected token sequence"
        )

    return {
        "target_tokens": target_tokens,
        "actual_tokens": len(verified_ids),
        "prompt": prompt,
        "token_ids": verified_ids,
    }


result = {
    "tokenizer_name_or_path": tokenizer.name_or_path,
    "tokenizer_class": type(tokenizer).__name__,
    "bos_token_id": tokenizer.bos_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "prompts": [
        make_exact_prompt(target)
        for target in configuration["targets"]
    ],
}

json.dump(result, sys.stdout)
"""


def construct_exact_prompts(
    *,
    image: str,
    cache_root: Path,
    model_snapshot_path: Path,
    targets: tuple[int, ...],
) -> dict:
    configuration = {
        "model_path": str(model_snapshot_path),
        "targets": list(targets),
    }

    with tempfile.TemporaryDirectory(prefix="dgx-prompt-") as temporary_directory:
        temporary_path = Path(temporary_directory)
        script_path = temporary_path / "construct_prompts.py"
        configuration_path = temporary_path / "configuration.json"

        script_path.write_text(TOKENIZER_SCRIPT)
        configuration_path.write_text(json.dumps(configuration))

        container_directory = "/tmp/dgx-prompt"

        command = [
            "docker",
            "run",
            "--rm",
            "--network",
            "none",
            "--entrypoint",
            "/bin/sh",
            "-v",
            f"{cache_root}:{cache_root}:ro",
            "-v",
            f"{temporary_path}:{container_directory}:ro",
            image,
            "-lc",
            (
                "python3 "
                f"{container_directory}/construct_prompts.py "
                f"{container_directory}/configuration.json"
            ),
        ]

        completed = subprocess.run(
            command,
            text=True,
            capture_output=True,
            timeout=REQUEST_TIMEOUT_S,
            check=False,
        )

    if completed.returncode != 0:
        raise RuntimeError(
            "Tokenizer container failed.\n\n"
            f"stdout:\n{completed.stdout}\n\n"
            f"stderr:\n{completed.stderr}"
        )

    return json.loads(completed.stdout)


prompt_construction = construct_exact_prompts(
    image=VLLM_IMAGE,
    cache_root=HF_CACHE_ROOT,
    model_snapshot_path=MODEL_SNAPSHOT_PATH,
    targets=PROMPT_TOKEN_COUNTS,
)

prompt_construction_summary = pd.DataFrame(
    [
        {
            "prompt_id": f"prompt_{item['target_tokens']}",
            "target_tokens": item["target_tokens"],
            "actual_tokens": item["actual_tokens"],
            "character_count": len(item["prompt"]),
            "token_id_checksum": sum(item["token_ids"]),
        }
        for item in prompt_construction["prompts"]
    ]
)

prompt_construction_summary

,prompt_id,target_tokens,actual_tokens,character_count,token_id_checksum
0,prompt_32,32,32,167,738859
1,prompt_512,512,512,3158,12317405


In [47]:
PROMPTS = {
    f"prompt_{item['target_tokens']}": item["prompt"]
    for item in prompt_construction["prompts"]
}

PROMPT_TOKEN_IDS = {
    f"prompt_{item['target_tokens']}": tuple(item["token_ids"])
    for item in prompt_construction["prompts"]
}

expected_prompt_ids = {f"prompt_{target}" for target in PROMPT_TOKEN_COUNTS}

if set(PROMPTS) != expected_prompt_ids:
    raise AssertionError(f"Unexpected prompt IDs: {sorted(PROMPTS)}")

for target in PROMPT_TOKEN_COUNTS:
    prompt_id = f"prompt_{target}"
    actual = len(PROMPT_TOKEN_IDS[prompt_id])

    if actual != target:
        raise AssertionError(f"{prompt_id}: expected {target} tokens, found {actual}")

print(f"Tokenizer class: {prompt_construction['tokenizer_class']}")
print(f"Tokenizer snapshot: {prompt_construction['tokenizer_name_or_path']}")
print(f"Registered prompts: {sorted(PROMPTS)}")

prompt_construction_summary

Tokenizer class: GemmaTokenizer
Tokenizer snapshot: /home/coert/.cache/huggingface/hub/models--google--gemma-4-E4B-it/snapshots/ee0ef6023621cff504d758262d4e04895a5af4a2
Registered prompts: ['prompt_32', 'prompt_512']


,prompt_id,target_tokens,actual_tokens,character_count,token_id_checksum
0,prompt_32,32,32,167,738859
1,prompt_512,512,512,3158,12317405


### Server lifecycle

For notebook-managed mode, construct an argument list only after selecting a model. Retain the exact `Popen` handle and terminate only that process during cleanup.

In [48]:
def inspect_image_configuration(
    image: str,
) -> dict[str, object]:
    completed = subprocess.run(
        [
            "docker",
            "image",
            "inspect",
            image,
            "--format",
            "{{json .Config}}",
        ],
        text=True,
        capture_output=True,
        timeout=30.0,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            "Could not inspect Docker image configuration.\n\n"
            f"stdout:\n{completed.stdout}\n\n"
            f"stderr:\n{completed.stderr}"
        )

    configuration = json.loads(completed.stdout)

    return {
        "image": image,
        "entrypoint": configuration.get("Entrypoint"),
        "cmd": configuration.get("Cmd"),
        "working_directory": configuration.get("WorkingDir"),
    }


image_configuration = inspect_image_configuration(VLLM_IMAGE)

pd.Series(image_configuration, name="value")

image                                         vllm-node
entrypoint           [/opt/nvidia/nvidia_entrypoint.sh]
cmd                                                None
working_directory                       /workspace/vllm
Name: value, dtype: object

In [49]:
import shlex
import subprocess
import uuid


SERVER_CONTAINER_NAME = f"dgx-spark-single-node-{uuid.uuid4().hex[:8]}"

SERVER_LOG_PATH = (
    repository_root
    / "experiments"
    / "01-distributed-inference"
    / f"{SERVER_CONTAINER_NAME}.log"
)

# Conservative initial envelope for the single-request baseline.
#
# DGX Spark exposes unified CPU/GPU memory, so leave substantial host headroom.
VLLM_GPU_MEMORY_UTILIZATION = 0.20

# The measured workload needs at most 512 prompt + 128 generated tokens.
# 1024 preserves debugging headroom without exposing the model's very large
# tokenizer/model limit to this first experiment.
VLLM_MAX_MODEL_LEN = 1_024

server_process: subprocess.Popen[str] | None = None
server_log_file = None
server_started_monotonic_s: float | None = None


def build_server_arguments(
    *,
    image: str,
    container_name: str,
    model_snapshot_path: Path,
    served_model_name: str,
    port: int,
    gpu_memory_utilization: float,
    max_model_len: int,
) -> list[str]:
    if not image:
        raise ValueError("image must not be empty")
    if not container_name:
        raise ValueError("container_name must not be empty")
    if not model_snapshot_path.is_dir():
        raise FileNotFoundError(model_snapshot_path)
    if not served_model_name:
        raise ValueError("served_model_name must not be empty")
    if not 1 <= port <= 65535:
        raise ValueError("port must be between 1 and 65535")
    if not 0 < gpu_memory_utilization <= 1:
        raise ValueError("gpu_memory_utilization must be in (0, 1]")

    required_model_len = max(PROMPT_TOKEN_COUNTS) + max(GENERATED_TOKEN_COUNTS)
    if max_model_len < required_model_len:
        raise ValueError(
            f"max_model_len={max_model_len} is smaller than the "
            f"largest benchmark sequence, {required_model_len}"
        )

    cache_root = Path.home() / ".cache" / "huggingface"

    return [
        "docker",
        "run",
        "--rm",
        "--name",
        container_name,
        "--gpus",
        "all",
        "--ipc",
        "host",
        "--network",
        "host",
        "-v",
        f"{cache_root}:{cache_root}:ro",
        image,
        "vllm",
        "serve",
        str(model_snapshot_path),
        "--served-model-name",
        served_model_name,
        "--host",
        HOST,
        "--port",
        str(port),
        "--gpu-memory-utilization",
        str(gpu_memory_utilization),
        "--max-model-len",
        str(max_model_len),
    ]


SERVER_ARGUMENTS = build_server_arguments(
    image=VLLM_IMAGE,
    container_name=SERVER_CONTAINER_NAME,
    model_snapshot_path=MODEL_SNAPSHOT_PATH,
    served_model_name=MODEL,
    port=PORT,
    gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    max_model_len=VLLM_MAX_MODEL_LEN,
)

pd.Series(
    {
        "container_name": SERVER_CONTAINER_NAME,
        "log_path": str(SERVER_LOG_PATH),
        "gpu_memory_utilization": VLLM_GPU_MEMORY_UTILIZATION,
        "max_model_len": VLLM_MAX_MODEL_LEN,
        "maximum_benchmark_sequence_tokens": (
            max(PROMPT_TOKEN_COUNTS) + max(GENERATED_TOKEN_COUNTS)
        ),
        "command": shlex.join(SERVER_ARGUMENTS),
    },
    name="value",
)

container_name                                          dgx-spark-single-node-68ca5e09
log_path                             /home/coert/workspace/dgx-spark-lab/experiment...
gpu_memory_utilization                                                             0.2
max_model_len                                                                     1024
maximum_benchmark_sequence_tokens                                                  640
command                              docker run --rm --name dgx-spark-single-node-6...
Name: value, dtype: object

In [50]:
def inspect_actual_vllm_serve_help(
    image: str,
) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "--gpus",
            "all",
            image,
            "vllm",
            "serve",
            "--help",
        ],
        text=True,
        capture_output=True,
        timeout=60.0,
        check=False,
    )


actual_serve_help = inspect_actual_vllm_serve_help(VLLM_IMAGE)

pd.Series(
    {
        "returncode": actual_serve_help.returncode,
        "stdout_characters": len(actual_serve_help.stdout),
        "stderr_characters": len(actual_serve_help.stderr),
        "contains_served_model_name": (
            "--served-model-name"
            in (actual_serve_help.stdout + actual_serve_help.stderr)
        ),
        "contains_host": (
            "--host" in (actual_serve_help.stdout + actual_serve_help.stderr)
        ),
        "contains_port": (
            "--port" in (actual_serve_help.stdout + actual_serve_help.stderr)
        ),
    },
    name="value",
)

returncode                        0
stdout_characters              5308
stderr_characters                 0
contains_served_model_name    False
contains_host                 False
contains_port                 False
Name: value, dtype: object

In [51]:
if actual_serve_help.returncode != 0:
    print("stdout:")
    print(actual_serve_help.stdout[-8_000:])
    print("\nstderr:")
    print(actual_serve_help.stderr[-8_000:])

    raise RuntimeError("`vllm serve --help` failed inside the pinned image")

print("The actual `vllm serve` command is available.")

The actual `vllm serve` command is available.


In [52]:
def inspect_named_container(
    container_name: str,
) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        [
            "docker",
            "inspect",
            container_name,
            "--format",
            "{{json .State}}",
        ],
        text=True,
        capture_output=True,
        timeout=10.0,
        check=False,
    )


existing_container = inspect_named_container(SERVER_CONTAINER_NAME)

if existing_container.returncode == 0:
    raise RuntimeError(
        f"Container {SERVER_CONTAINER_NAME!r} already exists; "
        "refusing to replace or stop it implicitly"
    )

if SERVER_LOG_PATH.exists():
    raise FileExistsError(f"Refusing to overwrite existing log: {SERVER_LOG_PATH}")

print(f"Container name is unused: {SERVER_CONTAINER_NAME}")
print(f"Log path is unused: {SERVER_LOG_PATH}")

Container name is unused: dgx-spark-single-node-68ca5e09
Log path is unused: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference/dgx-spark-single-node-68ca5e09.log


In [53]:
import time

if SERVER_MODE != "notebook_subprocess":
    raise RuntimeError("This cell requires SERVER_MODE='notebook_subprocess'")

if server_process is not None:
    raise RuntimeError("This notebook kernel already holds a server process handle")

server_log_file = SERVER_LOG_PATH.open(
    "w",
    encoding="utf-8",
    buffering=1,
)

server_started_monotonic_s = time.monotonic()

try:
    server_process = subprocess.Popen(
        SERVER_ARGUMENTS,
        stdin=subprocess.DEVNULL,
        stdout=server_log_file,
        stderr=subprocess.STDOUT,
        text=True,
        start_new_session=True,
    )
except Exception:
    server_log_file.close()
    server_log_file = None
    server_started_monotonic_s = None
    raise

print(f"Started Docker client process PID: {server_process.pid}")
print(f"Container name: {SERVER_CONTAINER_NAME}")
print(f"Server log: {SERVER_LOG_PATH}")

Started Docker client process PID: 3924541
Container name: dgx-spark-single-node-68ca5e09
Server log: /home/coert/workspace/dgx-spark-lab/experiments/01-distributed-inference/dgx-spark-single-node-68ca5e09.log


In [54]:
INITIAL_LAUNCH_OBSERVATION_S = 5.0

time.sleep(INITIAL_LAUNCH_OBSERVATION_S)

server_returncode = server_process.poll()

server_log_file.flush()
server_log_text = SERVER_LOG_PATH.read_text(
    encoding="utf-8",
    errors="replace",
)

launch_summary = pd.Series(
    {
        "docker_client_pid": server_process.pid,
        "docker_client_returncode": server_returncode,
        "observation_interval_s": INITIAL_LAUNCH_OBSERVATION_S,
        "log_characters": len(server_log_text),
        "container_name": SERVER_CONTAINER_NAME,
        "server_endpoint": ENDPOINT,
    },
    name="value",
)

launch_summary

docker_client_pid                                  3924541
docker_client_returncode                              None
observation_interval_s                                 5.0
log_characters                                         523
container_name              dgx-spark-single-node-68ca5e09
server_endpoint                      http://127.0.0.1:8000
Name: value, dtype: object

In [55]:
LOG_TAIL_CHARACTERS = 8_000

print(server_log_text[-LOG_TAIL_CHARACTERS:])

if server_returncode is not None:
    raise RuntimeError(
        "The vLLM container exited during initial launch. "
        f"Docker client return code: {server_returncode}. "
        "Inspect the saved log above."
    )

print(
    "The container remained active through the initial "
    f"{INITIAL_LAUNCH_OBSERVATION_S:.1f} s observation interval."
)


== CUDA ==

CUDA Version 13.0.2

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning-container-license

A copy of this license is made available in this container at /NGC-DL-CONTAINER-LICENSE for your convenience.


The container remained active through the initial 5.0 s observation interval.


### Readiness and request scaffolds

Poll the notebook-managed server until its OpenAI-compatible model endpoint is ready, while also detecting an early server-process exit.

In [56]:
import json
import time
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def request_json(
    url: str,
    *,
    timeout_s: float,
) -> tuple[int, dict[str, Any], dict[str, str]]:
    request = Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "dgx-spark-lab/01-single-node-inference",
        },
        method="GET",
    )

    with urlopen(request, timeout=timeout_s) as response:
        status_code = response.status
        headers = {key.lower(): value for key, value in response.headers.items()}
        body = response.read().decode("utf-8")

    parsed = json.loads(body)
    if not isinstance(parsed, dict):
        raise TypeError(
            f"Expected a JSON object from {url}, received {type(parsed).__name__}"
        )

    return status_code, parsed, headers


def wait_until_ready(
    endpoint: str,
    timeout_s: float,
    *,
    poll_interval_s: float = 1.0,
    request_timeout_s: float = 5.0,
    process: subprocess.Popen[str] | None = None,
) -> dict[str, Any]:
    """Poll the OpenAI-compatible model-list endpoint until it succeeds."""

    if timeout_s <= 0:
        raise ValueError("timeout_s must be positive")
    if poll_interval_s <= 0:
        raise ValueError("poll_interval_s must be positive")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be positive")

    models_url = f"{endpoint.rstrip('/')}/v1/models"
    started_s = time.monotonic()
    deadline_s = started_s + timeout_s

    attempts: list[dict[str, Any]] = []

    while True:
        if process is not None and process.poll() is not None:
            return {
                "ready": False,
                "models_url": models_url,
                "elapsed_s": time.monotonic() - started_s,
                "attempt_count": len(attempts),
                "advertised_models": (),
                "response_payload": None,
                "response_headers": {},
                "attempts": attempts,
            }

        attempt_started_s = time.monotonic()

        try:
            status_code, payload, headers = request_json(
                models_url,
                timeout_s=request_timeout_s,
            )

            attempts.append(
                {
                    "attempt": len(attempts) + 1,
                    "elapsed_s": time.monotonic() - started_s,
                    "request_latency_s": (time.monotonic() - attempt_started_s),
                    "status_code": status_code,
                    "error_type": None,
                    "error": None,
                }
            )

            model_records = payload.get("data")
            if not isinstance(model_records, list):
                raise ValueError("The /v1/models response does not contain a data list")

            advertised_models = tuple(
                record.get("id")
                for record in model_records
                if isinstance(record, dict) and isinstance(record.get("id"), str)
            )

            return {
                "ready": True,
                "models_url": models_url,
                "elapsed_s": time.monotonic() - started_s,
                "attempt_count": len(attempts),
                "advertised_models": advertised_models,
                "response_payload": payload,
                "response_headers": headers,
                "attempts": attempts,
            }

        except HTTPError as error:
            error_body = error.read().decode(
                "utf-8",
                errors="replace",
            )
            error_type = type(error).__name__
            error_message = (
                f"HTTP {error.code}: {error.reason}; body={error_body[:500]!r}"
            )

        except (
            URLError,
            TimeoutError,
            ConnectionError,
            json.JSONDecodeError,
            OSError,
            TypeError,
            ValueError,
        ) as error:
            error_type = type(error).__name__
            error_message = str(error)

        attempts.append(
            {
                "attempt": len(attempts) + 1,
                "elapsed_s": time.monotonic() - started_s,
                "request_latency_s": (time.monotonic() - attempt_started_s),
                "status_code": None,
                "error_type": error_type,
                "error": error_message,
            }
        )

        now_s = time.monotonic()
        if now_s >= deadline_s:
            return {
                "ready": False,
                "models_url": models_url,
                "elapsed_s": now_s - started_s,
                "attempt_count": len(attempts),
                "advertised_models": (),
                "response_payload": None,
                "response_headers": {},
                "attempts": attempts,
            }

        time.sleep(
            min(
                poll_interval_s,
                max(0.0, deadline_s - now_s),
            )
        )

In [57]:
SERVER_STARTUP_TIMEOUT_S = 900.0

readiness_result = wait_until_ready(
    ENDPOINT,
    timeout_s=SERVER_STARTUP_TIMEOUT_S,
    poll_interval_s=2.0,
    request_timeout_s=5.0,
    process=server_process,
)

server_ready_monotonic_s = time.monotonic()

startup_duration_s = (
    server_ready_monotonic_s - server_started_monotonic_s
    if readiness_result["ready"]
    else None
)

readiness_summary = pd.Series(
    {
        "ready": readiness_result["ready"],
        "models_url": readiness_result["models_url"],
        "startup_duration_s": startup_duration_s,
        "readiness_attempt_count": readiness_result["attempt_count"],
        "advertised_models": readiness_result["advertised_models"],
        "configured_model": MODEL,
        "configured_model_advertised": (MODEL in readiness_result["advertised_models"]),
        "docker_client_returncode": server_process.poll(),
    },
    name="value",
)

readiness_summary

ready                                                     True
models_url                     http://127.0.0.1:8000/v1/models
startup_duration_s                                  245.133212
readiness_attempt_count                                    121
advertised_models                     (google/gemma-4-E4B-it,)
configured_model                         google/gemma-4-E4B-it
configured_model_advertised                               True
docker_client_returncode                                  None
Name: value, dtype: object

In [58]:
readiness_attempts = pd.DataFrame(readiness_result["attempts"])

if not readiness_result["ready"]:
    server_log_file.flush()
    failed_log_text = SERVER_LOG_PATH.read_text(
        encoding="utf-8",
        errors="replace",
    )

    display(readiness_attempts.tail(10))
    print(failed_log_text[-8_000:])

    raise RuntimeError(
        f"Server did not become ready within {SERVER_STARTUP_TIMEOUT_S:.0f} seconds"
    )

if MODEL not in readiness_result["advertised_models"]:
    raise RuntimeError(
        f"Configured model {MODEL!r} was not advertised. "
        f"Advertised models: "
        f"{readiness_result['advertised_models']!r}"
    )

print(f"Server startup duration: {startup_duration_s:.3f} s")
print(f"Advertised model verified: {MODEL}")

Server startup duration: 245.133 s
Advertised model verified: google/gemma-4-E4B-it


### Completion API contract probe

Before warm-up or benchmark measurement, send one excluded diagnostic request.

This probe verifies that:

- the serialized prompt is accepted by `/v1/completions`;
- the server does not add another special token;
- server-reported prompt tokens match the tokenizer-derived target;
- `ignore_eos` produces the requested output length;
- the configured served-model identity is returned.

The probe is diagnostic only. Its latency is not part of warm-up or steady-state results.

In [59]:
from urllib.error import HTTPError
from urllib.request import Request, urlopen


def send_completion_probe(
    *,
    endpoint: str,
    model: str,
    prompt: str,
    max_tokens: int,
    timeout_s: float,
    sampling: dict[str, object],
) -> dict[str, object]:
    completions_url = f"{endpoint.rstrip('/')}/v1/completions"

    payload = {
        "model": model,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": sampling["temperature"],
        "top_p": sampling["top_p"],
        "seed": sampling["seed"],
        "stream": False,
        # The prompt already contains the tokenizer's serialized chat template
        # and special tokens.
        "add_special_tokens": False,
        # This controlled length probe must not stop early at EOS.
        "ignore_eos": True,
    }

    request = Request(
        completions_url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Accept": "application/json",
            "Content-Type": "application/json",
            "User-Agent": "dgx-spark-lab/01-single-node-inference",
        },
        method="POST",
    )

    started_s = time.monotonic()

    try:
        with urlopen(request, timeout=timeout_s) as response:
            status_code = response.status
            response_body = response.read().decode("utf-8")
    except HTTPError as error:
        error_body = error.read().decode(
            "utf-8",
            errors="replace",
        )
        raise RuntimeError(
            f"Completion probe failed with HTTP {error.code}: "
            f"{error.reason}; body={error_body[:2_000]!r}"
        ) from error

    completed_s = time.monotonic()
    response_payload = json.loads(response_body)

    choices = response_payload.get("choices")
    if not isinstance(choices, list) or len(choices) != 1:
        raise ValueError(
            f"Expected exactly one completion choice, received {choices!r}"
        )

    choice = choices[0]
    usage = response_payload.get("usage")

    if not isinstance(choice, dict):
        raise TypeError("Completion choice is not a JSON object")
    if not isinstance(usage, dict):
        raise TypeError("Completion response does not contain usage data")

    return {
        "status_code": status_code,
        "response_id": response_payload.get("id"),
        "response_model": response_payload.get("model"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
        "finish_reason": choice.get("finish_reason"),
        "latency_s": completed_s - started_s,
        "output_text": choice.get("text"),
    }


PROBE_PROMPT_ID = "prompt_32"
PROBE_OUTPUT_TOKENS = 16

completion_probe = send_completion_probe(
    endpoint=ENDPOINT,
    model=MODEL,
    prompt=PROMPTS[PROBE_PROMPT_ID],
    max_tokens=PROBE_OUTPUT_TOKENS,
    timeout_s=REQUEST_TIMEOUT_S,
    sampling=SAMPLING,
)

pd.Series(completion_probe, name="value")

status_code                                                        200
response_id                                      cmpl-9566755867373b65
response_model                                   google/gemma-4-E4B-it
prompt_tokens                                                       32
completion_tokens                                                   16
total_tokens                                                        48
finish_reason                                                   length
latency_s                                                     1.471401
output_text           each input. This sequence dictates the struct...
Name: value, dtype: object

In [60]:
expected_probe_prompt_tokens = len(PROMPT_TOKEN_IDS[PROBE_PROMPT_ID])

probe_invariants = pd.Series(
    {
        "http_status_is_200": (completion_probe["status_code"] == 200),
        "model_matches": (completion_probe["response_model"] == MODEL),
        "prompt_tokens_match": (
            completion_probe["prompt_tokens"] == expected_probe_prompt_tokens
        ),
        "completion_tokens_match": (
            completion_probe["completion_tokens"] == PROBE_OUTPUT_TOKENS
        ),
        "total_tokens_match": (
            completion_probe["total_tokens"]
            == expected_probe_prompt_tokens + PROBE_OUTPUT_TOKENS
        ),
        "finish_reason_is_length": (completion_probe["finish_reason"] == "length"),
        "output_is_nonempty": bool(completion_probe["output_text"]),
    },
    name="passed",
)

display(probe_invariants.to_frame())

if not bool(probe_invariants.all()):
    raise AssertionError(
        "The completion API contract probe failed one or more invariants"
    )

print("Completion API contract verified.")

,passed
http_status_is_200,True
model_matches,True
prompt_tokens_match,True
completion_tokens_match,True
total_tokens_match,True
finish_reason_is_length,True
output_is_nonempty,True


Completion API contract verified.


In [ ]:
def stop_notebook_server(
    *,
    container_name: str,
    process: subprocess.Popen[str] | None,
    log_file,
    timeout_s: float = 30.0,
) -> dict[str, object]:
    cleanup_started_s = time.monotonic()

    stop_result = subprocess.run(
        [
            "docker",
            "stop",
            "--time",
            str(int(timeout_s)),
            container_name,
        ],
        text=True,
        capture_output=True,
        timeout=timeout_s + 10.0,
        check=False,
    )

    process_returncode = None

    if process is not None:
        try:
            process_returncode = process.wait(timeout=10.0)
        except subprocess.TimeoutExpired:
            process.terminate()
            try:
                process_returncode = process.wait(timeout=5.0)
            except subprocess.TimeoutExpired:
                process.kill()
                process_returncode = process.wait(timeout=5.0)

    if log_file is not None and not log_file.closed:
        log_file.flush()
        log_file.close()

    return {
        "container_name": container_name,
        "docker_stop_returncode": stop_result.returncode,
        "docker_stop_stdout": stop_result.stdout.strip(),
        "docker_stop_stderr": stop_result.stderr.strip(),
        "process_returncode": process_returncode,
        "cleanup_duration_s": (time.monotonic() - cleanup_started_s),
    }


print(
    "Cleanup helper defined. Do not run it until measurements "
    "are complete or the server must be stopped."
)

### Warm-up and raw result schema

In [ ]:
WARMUP_RESULTS = []  # Run separately; never mix with measured trials.

raw_result_columns = (
    "request_id",
    "trial",
    "prompt_id",
    "prompt_tokens",
    "requested_output_tokens",
    "returned_output_tokens",
    "started_monotonic_s",
    "first_token_monotonic_s",
    "completed_monotonic_s",
    "ttft_s",
    "latency_s",
    "memory_bytes",
    "output_text",
    "output_correct",
    "status",
    "error",
)
raw_results = pd.DataFrame(columns=raw_result_columns)
raw_results

### Aggregation schema

In [ ]:
aggregate_columns = (
    "configuration",
    "prompt_tokens",
    "requested_output_tokens",
    "metric",
    "minimum",
    "median",
    "p90",
    "p99",
    "count",
    "failures",
)
aggregates = pd.DataFrame(columns=aggregate_columns)
aggregates

### Cleanup guidance

Stop only the `server_process` handle created by this notebook, first requesting graceful termination and then applying a bounded wait. Never use unscoped process-kill commands.

In [ ]:
cleanup_result = stop_notebook_server(
    container_name=SERVER_CONTAINER_NAME,
    process=server_process,
    log_file=server_log_file,
)

pd.Series(cleanup_result, name="value")

## Observations

TODO: Record only facts produced by the saved outputs of this notebook.

## Explanation

TODO: Explain the measured results. Separate derived values and architectural inference from direct observations.

## Connection to LLMs

TODO: Connect the verified result to inference behavior without claiming effects that were not measured.

## Further Exploration

TODO: Identify the next controlled experiment justified by the result.